# S09 — bounded multilingual reranking
Open one copy per configured worker (default two). This scores prepared dense/lexical candidates; it does not rebuild embeddings or run either retriever. Use distinct WORKER_ID values and worker folders. Never run two writers for the same worker. See docs/RETRIEVAL_STAGE9.md. CPU preparation/tests/baseline are completed locally; these GPU cells are owner-run; clinical QA and quality comparisons remain at S10.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
WORKER_ID = 0  # second session: 1; range depends on the frozen S09 config
BUNDLE = Path('/content/drive/MyDrive/ViBioMIR/stage9_rerank_colab.zip')
DRIVE_WORKERS = Path('/content/drive/MyDrive/ViBioMIR/stage9_rerank_workers')
PROJECT = Path('/content/vibiomir_stage9')
RUN_DIR = Path('/content') / f'stage9_worker_{WORKER_ID}'
BACKUP_DIR = DRIVE_WORKERS / f'worker_{WORKER_ID}'
BATCH_SIZE = None  # auto by VRAM; set an integer only to override
PADDED_TOKENS = None  # auto by VRAM; T4/L4/A100 presets
STOP_AFTER_QUERIES = 1  # then None; rerun worker cell to finish the partition
assert BUNDLE.is_file(), 'Upload the S09 ZIP and set BUNDLE first.'
os.environ.update(USE_TF='0', USE_FLAX='0', TOKENIZERS_PARALLELISM='false', OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1')


In [ ]:
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as archive:
    meta = json.loads(archive.read('BUNDLE.json'))
    canonical = json.dumps({k:v for k,v in meta.items() if k != 'signature'}, ensure_ascii=False, sort_keys=True, separators=(',', ':'))
    assert hashlib.sha256(canonical.encode()).hexdigest() == meta['signature'], 'Bundle metadata checksum differs'
    assert set(archive.namelist()) == set(meta['files']) | {'BUNDLE.json'}, 'Unexpected ZIP files'
    needed = sum(x.file_size for x in archive.infolist()) + 8 * 1024**3
    assert shutil.disk_usage('/content').free > needed, 'Insufficient local disk for packets, weights and checkpoints'
    for name, digest in meta['files'].items():
        path = (PROJECT / name).resolve()
        assert PROJECT.resolve() in path.parents, 'Unsafe ZIP path'
        data = archive.read(name)
        assert hashlib.sha256(data).hexdigest() == digest, f'ZIP payload changed: {name}'
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(data)
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-stage9-colab.txt'], check=True)
if str(PROJECT) not in sys.path: sys.path.insert(0, str(PROJECT))


In [ ]:
import torch
from transformers import AutoTokenizer
from vibiomir.stage9_pipeline import prepared, load_packet
from vibiomir.stage9_reranker import pair_windows, recommended_runtime
state = prepared(PROJECT)
workers = state['identity']['settings']['workers']
assert 0 <= WORKER_ID < workers, f'WORKER_ID must be 0..{workers-1}'
assert torch.cuda.is_available(), 'Select a CUDA GPU runtime before reranking'
policy = state['identity']['settings']['reranker']
gpu_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
preset = recommended_runtime(gpu_gib)
if BATCH_SIZE is None: BATCH_SIZE = preset['batch_size']
if PADDED_TOKENS is None: PADDED_TOKENS = preset['padded_tokens']
assert BATCH_SIZE > 0 and PADDED_TOKENS >= policy['max_pair_tokens']
assigned = [x for x in state['packets'] if x['worker_id'] == WORKER_ID]
assert assigned, 'This worker has no queries; use a full prepared run or fewer workers for the pilot'
print('GPU:', torch.cuda.get_device_name(0), round(gpu_gib, 1), 'GiB')
print('Runtime budgets:', BATCH_SIZE, 'pairs,', PADDED_TOKENS, 'padded tokens')
print('Assigned queries:', len(assigned), 'candidate pairs:', sum(x['candidates'] for x in assigned))
tokenizer = AutoTokenizer.from_pretrained(policy['model_id'], revision=policy['revision'], trust_remote_code=False, use_fast=True)
example = next((x for x in assigned if x['candidates']), None)
if example:
    packet = load_packet(PROJECT, example)
    windows = pair_windows(tokenizer, packet['query'], packet['candidates'][0]['retrieval_text'], policy)
    print('Real fast-tokenizer pair preflight passed; mapped windows:', len(windows))


In [ ]:
from datetime import datetime, timezone
LOG_PATH = PROJECT / f"worker_{WORKER_ID}_{datetime.now(timezone.utc):%Y%m%dT%H%M%S}.log"
BACKUP_DIR.mkdir(parents=True, exist_ok=True)
command = [sys.executable, '-u', '-m', 'vibiomir.stage9_pipeline', 'rerank-worker',
    '--prepared-dir', str(PROJECT), '--output-dir', str(RUN_DIR), '--worker-id', str(WORKER_ID),
    '--backup-dir', str(BACKUP_DIR), '--batch-size', str(BATCH_SIZE), '--padded-tokens', str(PADDED_TOKENS)]
if STOP_AFTER_QUERIES is not None: command += ['--stop-after-queries', str(STOP_AFTER_QUERIES)]
print('Worker log:', LOG_PATH, flush=True)
with LOG_PATH.open('w', encoding='utf-8') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True); log.write(line); log.flush()
        result = process.wait()
    except BaseException:
        process.terminate()
        try: process.wait(timeout=30)
        except subprocess.TimeoutExpired: process.kill(); process.wait()
        raise
shutil.copy2(LOG_PATH, BACKUP_DIR / LOG_PATH.name)
if result:
    raise RuntimeError(f'Worker exited {result}; the original error is above and saved in {LOG_PATH}. Committed queries are preserved.')
manifest = json.loads((RUN_DIR / 'manifest.json').read_text())
print('Worker state:', manifest['state'])
if manifest['state'] != 'complete':
    print('Set STOP_AFTER_QUERIES=None and rerun this cell; completed queries are verified and reused.')


In [ ]:
manifest = json.loads((RUN_DIR / 'manifest.json').read_text())
if manifest['state'] == 'complete':
    ARCHIVE = Path('/content') / f'stage9_worker_{WORKER_ID}.zip'
    subprocess.run([sys.executable, '-m', 'vibiomir.stage9_pipeline', 'archive-worker',
        '--prepared-dir', str(PROJECT), '--output-dir', str(RUN_DIR), '--worker-id', str(WORKER_ID),
        '--archive', str(ARCHIVE)], check=True)
    shutil.copy2(RUN_DIR / 'VALIDATION.json', BACKUP_DIR / 'VALIDATION.json')
    DRIVE_ARCHIVE = DRIVE_WORKERS / ARCHIVE.name
    pending = DRIVE_ARCHIVE.with_suffix('.zip.tmp')
    shutil.copy2(ARCHIVE, pending)
    pending.replace(DRIVE_ARCHIVE)
    print('Download this one worker ZIP:', DRIVE_ARCHIVE)
else:
    print('Finish the assigned queries before validation/archiving.')


## Return to local CPU
Download `stage9_worker_0.zip` and `stage9_worker_1.zip` (or all configured worker ZIPs) from the Drive `stage9_rerank_workers/` folder into local `artifacts/stage9_rerank_workers/`. Keep their names. Then run `python -m vibiomir.stage9_pipeline finish` from the local repository. It restores/validates workers, exports reranked results and checks the selected text against the original stage-5 source. Model weights, score-cache SQLite and full corpus folders are not in these ZIPs. Full-folder download is still supported if you retain the worker subdirectories. See docs/RETRIEVAL_STAGE9.md.
